# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We will load metadata, overview record sets and fields by their `@id`, extract data, and perform exploratory analyses, following best practices for dataset FAIRness.

### Dataset Source

The dataset [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) is provided via a public URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading

Load the metadata and records from the dataset Croissant schema URL using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

List all available record sets, fields, and columns, referencing their `@id`.
This information helps select specific subsets of the data for further exploration.

In [ ]:
# List all record sets and their fields/columns by @id
print("Record Sets with their '@id' and their fields/columns:")
record_sets_info = []
for rs in dataset.record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields in this record set:")
    for field in rs.fields:
        print(f"    - name: {field.name}, @id: {field.id}")
    columns = getattr(rs, 'columns', [])
    if columns:
        print(f"  Columns:")
        for col in columns:
            print(f"    - name: {col.name}, @id: {col.id}")    print()
    record_sets_info.append({
        "name": rs.name,
        "id": rs.id,
        "field_ids": [f.id for f in rs.fields],
        "column_ids": [c.id for c in getattr(rs, 'columns', [])]
    })

## 3. Data Extraction

Load records from specific record sets into pandas DataFrames using their `@id`. Reference fields/columns using their `@id` as well.

In [ ]:
# ---
# Select the main record set to load. If multiple record sets in the data, adjust accordingly.
# For this dataset, the main tabular record set for case-level data usually has a name like "Clinical records",
# but printouts above will show the exact @id(s).
# Replace the following list with the actual @id(s) from the previous cell if they differ.

# Example: assuming only one main record set, grabbing the first
record_sets = [rs['id'] for rs in record_sets_info]
dataframes = {}

for record_set_id in record_sets:
    print(f"\nLoading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns:", df.columns.tolist())
    else:
        print("No records found for this record set.")

# Display top rows of the main DataFrame (use first record set as example)
if record_sets:
    main_record_set = record_sets[0]
    if main_record_set in dataframes:
        display_cols = dataframes[main_record_set].columns.tolist()
        print(f"\nColumns for record set {main_record_set}:\n", display_cols)
        dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps, such as filtering records based on specific criterion, normalizing numeric fields, and grouping or categorizing data.

**Note:** All references use `@id` for record sets and fields.

In [ ]:
# Pick a numeric field for demonstration
# Inspect columns in your DataFrame. For this dataset, expected numeric fields
# might include: 'cr:age', 'cr:interval_months', etc. Adjust according to results above.
main_df = dataframes[main_record_set]
numeric_field = None
# Pick column whose type is numeric if available
for col in main_df.columns:
    # crude test: is type int or float?
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field = col
        break

if numeric_field is None:
    # Default to a candidate column name (replace as required)
    numeric_field = main_df.columns[0]  # fallback

print(f"Using numeric field for example analysis: {numeric_field}")

# Filter records (adjust threshold for your field; here using median as threshold)
if pd.api.types.is_numeric_dtype(main_df[numeric_field]):
    threshold = main_df[numeric_field].median()
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    print(f"Field {numeric_field} is not numeric. Please choose a numeric field for EDA.")

# Try grouping, e.g., by an anatomical location or categorical @id if present
group_field = None
for col in main_df.columns:
    # Pick a non-numeric category field
    if not pd.api.types.is_numeric_dtype(main_df[col]) and main_df[col].nunique() < 10:
        group_field = col
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable categorical field for grouping found.")

## 5. Visualization

Visualize the distribution of a selected numeric field and its relationship to a categorical field (if available), using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If grouping field available, show boxplot
if group_field is not None:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=main_df, y=group_field, x=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion

- The FAIR² dataset was loaded and explored using `mlcroissant`.
- Record sets, fields, and columns were accessed by their `@id`, supporting reproducible referencing.
- Common EDA steps, filtering, normalization, grouping, and basic visualization illustrated how to process structured data from Croissant schemas for clinical/biomedical research.

You can now adapt this notebook for more advanced analyses or to reference other Croissant-compliant biomedical datasets!